# Воспроизведение обучения армянской GPT

Это последовательные вызовы полного кода data / tokenizer / train.
Все веса создаются случайно; финальная модель получается из нашей 97M после continued pretraining.
Ноутбук не содержит сохранённых выводов повторного обучения. Исходные результаты находятся в отчёте.
Запускать после установки пакета из корня репозитория, на GPU с BF16. Настройки ниже совпадают с сохранёнными конфигурациями.

In [ ]:
from pathlib import Path
import os
import json

os.environ.setdefault("LLM_STORAGE_ROOT", "/data/armenian_llm")
from armenian_llm.paths import DATA_DIR, CHECKPOINTS_DIR, TOKENIZER_500K_DATA_DIR, RUN_CONFIGS_DIR
print("Хранилище:", os.environ["LLM_STORAGE_ROOT"])

## 1. ArmWeb
Скачиваю фиксированную версию, смотрю выборку и применяю сохранённые правила очистки.

In [ ]:
%run -m armenian_llm.data.armweb.download
%run -m armenian_llm.data.armweb.inspect
%run -m armenian_llm.data.armweb.analyze
%run -m armenian_llm.data.armweb.repetition

In [ ]:
%run -m armenian_llm.data.armweb.prepare

## 2. BPE и поток токенов
BPE 16k обучается на 500k документов train. Validation содержит 5k документов и не используется для обучения токенизатора.

In [ ]:
from armenian_llm.tokenization.sample import main as prepare_tokenizer_sample
from armenian_llm.tokenization.train import run as train_bpe

prepare_tokenizer_sample(output_dir=TOKENIZER_500K_DATA_DIR, train_size=500_000, validation_size=5_000, seed=42)
train_bpe(TOKENIZER_500K_DATA_DIR, vocab_size=16_000)

In [ ]:
%run -m armenian_llm.data.encode

## 3. Исходная 97M
Конфигурация берётся из фактического запуска: 12 слоёв, ширина 768, 2 эпохи, effective batch 128.
Для продолжения прерванного запуска повтори вызов с resume=True.

In [ ]:
from armenian_llm.modeling.gpt import GPTConfig
from armenian_llm.training.pretrain import TrainConfig, main as pretrain

base_config = json.loads((RUN_CONFIGS_DIR / "arm_gpt_97m.json").read_text(encoding="utf-8"))
pretrain(
    run_dir=CHECKPOINTS_DIR / "arm_gpt_97m",
    model_config=GPTConfig(**base_config["model"]),
    train_config=TrainConfig(**base_config["training"]),
    resume=False,
)

## 4. Wikipedia
Загрузка, анализ и очистка выборки идут до подготовки всех статей. Просмотренные статьи остаются только в train.

In [ ]:
%run -m armenian_llm.data.wikipedia.download
%run -m armenian_llm.data.wikipedia.inspect
%run -m armenian_llm.data.wikipedia.clean
%run -m armenian_llm.data.wikipedia.filter
%run -m armenian_llm.data.wikipedia.prepare

In [ ]:
from armenian_llm.data.encode import main as encode
encode(
    input_dir=DATA_DIR / "processed" / "wikipedia_hy_20231101",
    output_dir=DATA_DIR / "tokenized" / "wikipedia_hy_20231101" / "bpe_16000_500000",
    splits=("train", "validation", "test"),
)

## 5. Продолжение претрейна
Смесь задаётся по целевым токенам: 70% ArmWeb, 30% Wikipedia.
Новый AdamW, 5000 шагов, LR 1e-4 → 1e-5. Начальные веса — best.pt нашей исходной 97M.

In [ ]:
%run -m armenian_llm.data.mixture
%run -m armenian_llm.training.continue_pretrain --variant mix --stop-at-step 5000

## 6. Отложенные тексты и вопросы
Checkpoint уже выбран на validation. Test loss и QA не меняют веса.

In [ ]:
%run -m armenian_llm.evaluation.final

from armenian_llm.evaluation.facts import main as evaluate_facts
evaluate_facts(models=["arm_gpt_97m", "arm_gpt_97m_cpt_mix"], zero_shot=True,
               output_name="armenian_facts_cpt_mix_zero_shot")

## 7. Экспорт
Экспортируется готовый best.pt финальной смеси вместе с токенизатором и кодом. GPU для упаковки не используется.

In [ ]:
%run -m armenian_llm.reporting.export_model

## Дополнительные эксперименты
Конфигурации 35M, пилота 222M, контрольной новостной ветки и SFT сохранены в configs/runs/.
35M: train.main с соответствующими GPTConfig и TrainConfig.
222M: pretrain_222m.py — только пилот 2000 шагов.
Новостной контроль: continue_pretrain.py --variant news --stop-at-step 1000.
BPE 8k/32k: train_tokenizer.run на той же выборке 500k с другим vocab_size.
Для SFT используются download_sft_data.py, prepare_sft_data.py и train_sft.py.
Результаты и ограничения каждого эксперимента разобраны в 03_results.ipynb.